In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
save_path = '/content/drive/MyDrive/genAI'

In [6]:
import tensorflow as tf
import numpy as np
import pandas as pd

import os
import time
from tf_keras.models import load_model
import pickle

In [4]:
## loading all models

model = load_model(os.path.join(save_path,'model.h5'))

## load encoder and scalar
with open(os.path.join(save_path,'label_encoder_gender.pkl'),'rb') as file:
  label_encoder_gender = pickle.load(file)

with open(os.path.join(save_path,'onehot_encoder_geo.pkl'),'rb') as file:
  onehot_encoder_geo = pickle.load(file)

with open(os.path.join(save_path,'scaler.pkl'),'rb') as file:
  scaler = pickle.load(file)

In [9]:
# Example input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [10]:
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
geo_encoder = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoder_df = pd.DataFrame(geo_encoder, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoder_df.head()

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [11]:
# Concatenate
input_data_df = pd.concat([input_data_df.drop('Geography',axis=1),
                           geo_encoder_df], axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [12]:
input_data_scaled = scaler.transform(input_data_df)
input_data_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [13]:
prediction = model.predict(input_data_scaled)
prediction

1/1 [==============================] - 0s 182ms/step


array([[0.04927324]], dtype=float32)

In [15]:
if prediction[0][0] > 0.5:
  print('The Customer is likely to churn')
else:
  print('The Customer is not likely to churn')

The Customer is not likely to churn
